# 01 - Data Collection from PubChem

This notebook demonstrates how to collect chemical compound data from the PubChem database using the REST API.

**Author**: Computational Chemistry Research Group  
**Date**: 2024  
**Purpose**: Collect molecular structures and properties for ML training

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.pubchem_collector import PubChemCollector
from src.data.preprocessor import DataPreprocessor
from src.utils.config import settings

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

print(f"Project root: {project_root}")
print(f"Data directory: {settings.RAW_DATA_DIR}")

## 2. Collect Compounds from PubChem

We use the PubChem PUG REST API to collect compound data. The collector fetches:
- Molecular weight, LogP, TPSA
- Hydrogen bond donors/acceptors
- Canonical SMILES
- Additional physicochemical properties

In [ ]:
# Initialize collector with conservative settings
collector = PubChemCollector(
    batch_size=100,      # Compounds per request
    max_retries=3,       # Retry failed requests
    request_delay=0.2,   # Rate limiting (seconds)
)

print("PubChem Collector initialized")
print(f"  Batch size: {collector.batch_size}")
print(f"  Max retries: {collector.max_retries}")
print(f"  Request delay: {collector.request_delay}s")

In [ ]:
# Collect 200 compounds (adjust as needed)
# Note: Full dataset collection of 1000+ compounds takes ~10-15 minutes

df = collector.collect_compounds(
    n=200,
    min_mol_weight=50.0,
    max_mol_weight=1000.0,
    require_smiles=True,
)

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 3. Explore the Dataset

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Summary statistics
df.describe()

## 4. Visualize Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Molecular weight
axes[0, 0].hist(df['molecular_weight'], bins=30, color='#3498db', edgecolor='white')
axes[0, 0].set_title('Molecular Weight Distribution')
axes[0, 0].set_xlabel('Molecular Weight (g/mol)')

# LogP
if 'xlogp' in df.columns:
    axes[0, 1].hist(df['xlogp'].dropna(), bins=30, color='#e74c3c', edgecolor='white')
    axes[0, 1].set_title('LogP Distribution')
    axes[0, 1].set_xlabel('LogP')

# TPSA
if 'tpsa' in df.columns:
    axes[0, 2].hist(df['tpsa'].dropna(), bins=30, color='#2ecc71', edgecolor='white')
    axes[0, 2].set_title('TPSA Distribution')
    axes[0, 2].set_xlabel('TPSA (Å²)')

# HBD vs HBA
if 'hbd' in df.columns and 'hba' in df.columns:
    axes[1, 0].scatter(df['hbd'], df['hba'], alpha=0.5, c='#9b59b6')
    axes[1, 0].set_title('HBD vs HBA')
    axes[1, 0].set_xlabel('HBD')
    axes[1, 0].set_ylabel('HBA')

# Toxicity category
if 'toxicity_category' in df.columns:
    toxicity_counts = df['toxicity_category'].value_counts().sort_index()
    axes[1, 1].bar(toxicity_counts.index, toxicity_counts.values, color='#f39c12')
    axes[1, 1].set_title('Toxicity Category Distribution')
    axes[1, 1].set_xlabel('Toxicity Category')

# Boiling point
if 'boiling_point' in df.columns:
    axes[1, 2].hist(df['boiling_point'].dropna(), bins=30, color='#1abc9c', edgecolor='white')
    axes[1, 2].set_title('Boiling Point Distribution')
    axes[1, 2].set_xlabel('Boiling Point (°C)')

plt.tight_layout()
plt.savefig(settings.PROCESSED_DATA_DIR / 'data_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
numeric_cols = [c for c in numeric_cols if c not in ['cid']]

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax, annot=False)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(settings.PROCESSED_DATA_DIR / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save the Dataset

In [ ]:
# Save to CSV
output_path = settings.RAW_DATA_DIR / 'pubchem_compounds.csv'
df.to_csv(output_path, index=False)
print(f"Dataset saved to: {output_path}")
print(f"Total compounds: {len(df)}")
print(f"Total features: {len(df.columns)}")
print(f"\nTarget variables:")
for col in ['solubility', 'boiling_point', 'toxicity_category']:
    if col in df.columns:
        print(f"  {col}: {df[col].notna().sum()}/{len(df)} non-null")

## 7. Summary

In this notebook, we:
1. Initialized the PubChem data collector
2. Collected compound data including SMILES, molecular properties, and target values
3. Explored the dataset distributions
4. Analyzed feature correlations
5. Saved the dataset for further processing

**Next**: Go to `02_feature_engineering.ipynb` to compute molecular descriptors.